In [1]:
import asyncio
import os
from typing import Any

import nest_asyncio
from IPython.display import Markdown, display
from dotenv import load_dotenv

from helpers import authenticate

nest_asyncio.apply()

### Define BeeAI Components

In [2]:
from beeai_framework.adapters.a2a.agents import A2AAgent
from beeai_framework.adapters.vertexai import VertexAIChatModel
from beeai_framework.agents.requirement import RequirementAgent
from beeai_framework.agents.requirement.requirements.conditional import (
    ConditionalRequirement,
)
from beeai_framework.memory import UnconstrainedMemory
from beeai_framework.memory.unconstrained_memory import UnconstrainedMemory
from beeai_framework.middleware.trajectory import EventMeta, GlobalTrajectoryMiddleware
from beeai_framework.tools import Tool
from beeai_framework.tools.handoff import HandoffTool
from beeai_framework.tools.think import ThinkTool


class ConciseGlobalTrajectoryMiddleware(GlobalTrajectoryMiddleware):
    def _format_prefix(self, meta: EventMeta) -> str:
        prefix = super()._format_prefix(meta)
        return prefix.rstrip(": ")

    def _format_payload(self, value: Any) -> str:
        return ""

In [3]:
load_dotenv(override=True)
_, project_id = authenticate()

In [4]:
host = os.environ.get("AGENT_HOST")
policy_agent_port = os.environ.get("POLICY_AGENT_PORT")
research_agent_port = os.environ.get("RESEARCH_AGENT_PORT")
provider_agent_port = os.environ.get("PROVIDER_AGENT_PORT")
healthcare_agent_port = int(os.environ.get("HEALTHCARE_AGENT_PORT"))

## Crear Instancias de los Agentes

In [5]:
policy_agent = A2AAgent(
    url=f"http://{host}:{policy_agent_port}", 
    memory=UnconstrainedMemory()
)
# Run `check_agent_exists()` to fetch and populate AgentCard
asyncio.run(policy_agent.check_agent_exists())
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ InsurancePolicyCoverageAgent initialized


In [6]:
research_agent = A2AAgent(
    url=f"http://{host}:{research_agent_port}", 
    memory=UnconstrainedMemory()
)
asyncio.run(research_agent.check_agent_exists())
print("\tℹ️", f"{research_agent.name} initialized")

	ℹ️ HealthResearchAgent initialized


In [7]:
provider_agent = A2AAgent(
    url=f"http://{host}:{provider_agent_port}", 
    memory=UnconstrainedMemory()
)
asyncio.run(provider_agent.check_agent_exists())
print("\tℹ️", f"{provider_agent.name} initialized")

	ℹ️ HealthcareProviderAgent initialized


## Configure the Orchestrator (Healthcare Concierge)

In [8]:
import os
from google.auth import default

print("ENV GOOGLE_CLOUD_PROJECT:", os.getenv("GOOGLE_CLOUD_PROJECT"))
print("ENV GOOGLE_APPLICATION_CREDENTIALS:", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))

creds, detected_project = default()
print("ADC detected project:", detected_project)
print("Creds type:", type(creds).__name__)


ENV GOOGLE_CLOUD_PROJECT: a2a-protocol-488221
ENV GOOGLE_APPLICATION_CREDENTIALS: a2a-protocol-488221-de424eb7f166.json
ADC detected project: a2a-protocol-488221
Creds type: Credentials


In [9]:
healthcare_agent = RequirementAgent(
    name="Healthcare Agent",
    description="""A personal concierge for Healthcare Information, 
    customized to your policy.""",
    llm=VertexAIChatModel(
        model_id="gemini-2.5-pro",
        project=project_id,
        location="us-central1",
        allow_parallel_tool_calls=True,
        allow_prompt_caching=False,
    ),
    tools=[
        thinktool := ThinkTool(),
        policy_tool := HandoffTool(
            target=policy_agent,
            name=policy_agent.name,
            description=policy_agent.agent_card.description,
        ),
        research_tool := HandoffTool(
            target=research_agent,
            name=research_agent.name,
            description=research_agent.agent_card.description,
        ),
        provider_tool := HandoffTool(
            target=provider_agent,
            name=provider_agent.name,
            description=provider_agent.agent_card.description,
        ),
    ],
    requirements=[
        ConditionalRequirement(
            thinktool, force_at_step=1, force_after=Tool, 
            consecutive_allowed=False
        ),
    ],
    role="Healthcare Concierge",
    instructions=(
        f"""You are a concierge for healthcare services. Your task is 
        to handoff to one or more agents to answer questions and provide 
        a detailed summary of their answers. Be sure that all of their 
        questions are answered before responding.
        Use `{policy_agent.name}` to answer insurance-related questions.
        
        IMPORTANT: When returning answers about providers, only output 
        providers from `{provider_agent.name}` and only provide insurance 
        information based on the results from `{policy_agent.name}`.

        In your output, put which agent gave you the information!"""
    ),
)

print("[INFO]", f"{healthcare_agent.meta.name} initialized")

[INFO] Healthcare Agent initialized


## Run the Full Workflow

In [10]:
try:
    response = await healthcare_agent.run(
        """I'm based in Austin, TX. How do I get mental health therapy near me 
        and what does my insurance cover?"""
    ).middleware(ConciseGlobalTrajectoryMiddleware())
    display(Markdown(response.last_message.text))
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e)}")
    import traceback
    traceback.print_exc()

🤖 RequirementAgent[Healthcare Agent][start]
--> 🔎 ConditionalRequirement[ConditionThink][start]
<-- 🔎 ConditionalRequirement[ConditionThink][success]
--> 💬 VertexAIChatModel[VertexAIChatModel][start]
<-- 💬 VertexAIChatModel[VertexAIChatModel][success]
--> 🛠️ ThinkTool[think][start]
<-- 🛠️ ThinkTool[think][success]
--> 🔎 ConditionalRequirement[ConditionThink][start]
<-- 🔎 ConditionalRequirement[ConditionThink][success]
--> 💬 VertexAIChatModel[VertexAIChatModel][start]
<-- 💬 VertexAIChatModel[VertexAIChatModel][success]
--> 🛠️ HandoffTool[healthcareprovideragent][start]
    --> 🤖 A2AAgent[HealthcareProviderAgent][start]
    <-- 🤖 A2AAgent[HealthcareProviderAgent][success]
<-- 🛠️ HandoffTool[healthcareprovideragent][success]
--> 🔎 ConditionalRequirement[ConditionThink][start]
<-- 🔎 ConditionalRequirement[ConditionThink][success]
--> 💬 VertexAIChatModel[VertexAIChatModel][start]
<-- 💬 VertexAIChatModel[VertexAIChatModel][success]
--> 🛠️ ThinkTool[think][start]
<-- 🛠️ ThinkTool[think][succe

I have found a mental health provider for you in Austin, TX and some information about your insurance coverage. Please note that I was unable to retrieve the complete details of your insurance plan.

**Healthcare Provider**
*   **Name:** Dr. Jessica Coffey
*   **Specialty:** Psychiatry
*   **Address:** 1201 West 38th Street, Suite 210, Austin, TX 78705
*   **Phone:** (512) 555-0199
*   **Email:** j.coffey@austinmindhealth.com
*   **Years of Experience:** 13
*   **Accepts New Patients:** Yes
*   **Insurance Accepted:** Blue Cross Blue Shield, UnitedHealth, Aetna, Humana

*Information from HealthcareProviderAgent*

**Insurance Coverage**
Based on the information I could retrieve, here is a partial summary of your mental health coverage:

*   **Outpatient Services (like therapy sessions):**
    *   **In-Network Provider:** You will pay 10% coinsurance for office visits and other outpatient services after your deductible has been met.

*Information from InsurancePolicyCoverageAgent*

I hope this information is helpful. I recommend contacting your insurance provider directly for the complete and most up-to-date details on your mental health coverage.